# 20260106_E_2_1_FCMC — 3D reconstruction workflow

This notebook is the runbook for reconstructing **one experiment** with the current repo code:

`SLEAP inference` → `top-cam 2D ID` → `reconstruct_from_slp` → `QC`.

| Item | Path |
|---|---|
| Experiment | `20260106_E_2_1_FCMC` |
| Data root | `Z:\medaka` |
| Raw video | `Z:\medaka\medaka_202512\20260106_E_2_1_FCMC` |
| SLEAP models | `Z:\medaka\sleap_models` |
| SLEAP output | `Z:\medaka\sleap_output` |
| 3D output | `Z:\medaka\reconstruction_3d\20260106_E_2_1_FCMC\<round_tag>` |

Long GPU jobs are **off by default**. Set the `RUN_*` flags in the config cell, then run that stage.
Use the project `.venv` kernel. Do **not** use `--mode predict` / TensorRT on this machine.

## 0. Current disk status (inspected 2026-10-26)

Raw video is complete (348600 frames, 166 fps, three cameras).

Existing SLEAP output is **not enough for 3D**:

- `Z:\medaka\sleap_output\20260106_E_2_1_FCMC\round7_260704\` has only **top_cam chunk** `.slp` files
- no `side1_cam` / `side2_cam`
- no merged `<view>.slp`
- no 3D CSV under `Z:\medaka\reconstruction_3d`

Default model for this notebook: **round8** (`retrain_20260722_round8` / `round8_260723_strictmouth`).
That is the newest top-down mouth+tail model `scripts.sleap.run_round_inference` can launch.
August 3-node folders exist but use a different layout and are not used here.

`configs/paths.local.toml` still lists `Z:\StephensU\medaka`. Every command below passes `Z:\medaka` explicitly.

## 1. Config

Change `ROUND_TAG` to `round7_260704` only if you want to finish the existing top-camera chunks instead of running round8.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    for p in [ROOT, *ROOT.parents]:
        if (p / "pyproject.toml").exists():
            ROOT = p
            break
sys.path.insert(0, str(ROOT))

PY = ROOT / ".venv" / "Scripts" / "python.exe"
if not PY.exists():
    PY = Path(sys.executable)

EXP = "20260106_E_2_1_FCMC"
DATA_ROOT = Path(r"Z:\medaka\medaka_202512")
MODELS = Path(r"Z:\medaka\sleap_models")
SLP_ROOT = Path(r"Z:\medaka\sleap_output")
RECON_ROOT = Path(r"Z:\medaka\reconstruction_3d")

ROUND_DIR = MODELS / "retrain_20260722_round8"
ROUND_TAG = "round8_260723_strictmouth"
SLP_DIR = SLP_ROOT / EXP / ROUND_TAG
OUT_DIR = RECON_ROOT / EXP / ROUND_TAG

# Flip these one at a time. Full inference is ~3.2 h/view.
RUN_SMOKE_INFERENCE = False
RUN_FULL_INFERENCE = False
RUN_MERGE_CHUNKS = False  # only needed for HPC-style round7 chunks
RUN_OVERLAY_QC = False
RUN_TOP_TRACKS = False
RUN_RECON_SMOKE = False
RUN_RECON_FULL = False

INFER_VIEWS = ["top_cam", "side1_cam", "side2_cam"]
SMOKE_FRAMES = "0-1659"
RECON_SMOKE_END = 4999


def run_mod(module: str, args: list[str]) -> int:
    cmd = [str(PY), "-m", module, *args]
    print("$", subprocess.list2cmdline(cmd))
    completed = subprocess.run(cmd, cwd=str(ROOT))
    if completed.returncode != 0:
        raise SystemExit(f"{module} failed with exit {completed.returncode}")
    return completed.returncode


print("ROOT     ", ROOT)
print("python   ", PY)
print("experiment", EXP)
print("round    ", ROUND_TAG)
print("videos   ", DATA_ROOT / EXP)
print("slp dir  ", SLP_DIR)
print("3d dir   ", OUT_DIR)

## 2. Inventory — videos, models, existing `.slp`

In [ ]:
VIDEO = {
    "top_cam": DATA_ROOT / EXP / "top_cam" / "cam0_top.mp4",
    "side1_cam": DATA_ROOT / EXP / "side1_cam" / "cam1_side.mp4",
    "side2_cam": DATA_ROOT / EXP / "side2_cam" / "cam2_side.mp4",
}

print("=== raw videos ===")
for view, path in VIDEO.items():
    status = f"{path.stat().st_size / 1e9:.2f} GB" if path.exists() else "MISSING"
    print(f"  {view:10s} {status:12s} {path}")

meta = DATA_ROOT / EXP / "experiment_metadata.json"
if meta.exists():
    payload = json.loads(meta.read_text(encoding="utf-8"))
    print("\n=== metadata frames ===")
    for view, info in payload.get("views", {}).items():
        print(f"  {view:10s} frames={info.get('total_frames')} fps={info.get('frame_rate')}")

print("\n=== sleap models ===")
if MODELS.exists():
    for child in sorted(MODELS.iterdir()):
        if child.is_dir():
            print(" ", child.name)

print("\n=== existing sleap_output for this experiment ===")
exp_slp = SLP_ROOT / EXP
if not exp_slp.exists():
    print("  (none)")
else:
    for path in sorted(exp_slp.rglob("*")):
        if path.is_file():
            print(f"  {path.relative_to(exp_slp)}  ({path.stat().st_size / 1e6:.1f} MB)")

need = [SLP_DIR / f"{v}.slp" for v in ("top_cam", "side1_cam", "side2_cam")]
print("\n=== round files needed for 3D ===")
for path in need:
    print(("  OK  " if path.exists() else "  --  ") + str(path))
if not all(p.exists() for p in need):
    print("\n3D reconstruction cannot start until all three merged <view>.slp exist.")

## 3. SLEAP inference

Local torch path (`--mode track`, the default). Always pass `--max-instances 6`.
This writes **one merged** `<view>.slp` per camera under `sleap_output/<exp>/<round_tag>/`.
Chunk merge is not needed after a successful local run.

Expect ~3.2 h/view at full length. Logs stay header-only until a view finishes — watch for the `.slp` file.

In [ ]:
def infer_args(round_tag: str, extra: list[str] | None = None) -> list[str]:
    args = [
        "--round-dir", str(ROUND_DIR),
        "--round-tag", round_tag,
        "--experiment", EXP,
        "--data-root", str(DATA_ROOT),
        "--output-root", str(SLP_ROOT),
        "--predict-batch-size", "12",
        "--max-instances", "6",
        "--continue-on-fail",
    ]
    if extra:
        args.extend(extra)
    return args


if RUN_SMOKE_INFERENCE:
    run_mod(
        "scripts.sleap.run_round_inference",
        infer_args(f"{ROUND_TAG}_smoke", ["--frames", SMOKE_FRAMES]),
    )
elif RUN_FULL_INFERENCE:
    run_mod(
        "scripts.sleap.run_round_inference",
        infer_args(ROUND_TAG, ["--views", *INFER_VIEWS]),
    )
else:
    print("Inference skipped. Set RUN_SMOKE_INFERENCE or RUN_FULL_INFERENCE.")
    print("Smoke:")
    print("  RUN_SMOKE_INFERENCE = True   # ~10 s of video, all 3 views")
    print("Full:")
    print("  RUN_FULL_INFERENCE = True    # ~10 h total")

### Optional shortcut: finish round7 top chunks

Only if you change config to `ROUND_DIR = MODELS / "retrain_20260704"` and `ROUND_TAG = "round7_260704"`.
Then set `INFER_VIEWS = ["side1_cam", "side2_cam"]`, run full inference, and merge chunks:

In [ ]:
if RUN_MERGE_CHUNKS:
    run_mod(
        "scripts.tracking.main",
        [
            "--merge-chunks", EXP,
            "--paths-config", "configs/paths.local.toml",
            "--sleap-output-dir", str(SLP_ROOT),
            "--medaka-data-dir", str(DATA_ROOT),
            "--sleap-subdir", ROUND_TAG,
        ],
    )
else:
    print("Merge skipped (not needed after local round8 inference).")

## 4. Overlay QC

`visualize_predictions` has no `--sleap-subdir`; pass `--slp` explicitly.

In [ ]:
if RUN_OVERLAY_QC:
    for view in ("top_cam", "side1_cam", "side2_cam"):
        run_mod(
            "scripts.sleap.visualize_predictions",
            [
                EXP,
                "--paths-config", "configs/paths.local.toml",
                "--sleap-output-dir", str(SLP_ROOT),
                "--medaka-data-dir", str(DATA_ROOT),
                "--view", view,
                "--slp", str(SLP_DIR / f"{view}.slp"),
                "--seconds", "5",
                "--radius", "10",
            ],
        )
else:
    print("Overlay QC skipped. Set RUN_OVERLAY_QC = True after the three .slp exist.")

## 5. Top-camera 2D identity (`track_simple_nn`)

This CSV is the bind anchor for `final_fish_id`. Do **not** pass `--anchor-node`.

In [ ]:
TOP_TRACKS = SLP_DIR / "top_cam.tracks.csv"

if RUN_TOP_TRACKS:
    run_mod(
        "scripts.tracking.track_simple_nn",
        [
            EXP,
            "--paths-config", "configs/paths.local.toml",
            "--sleap-output-dir", str(SLP_ROOT),
            "--view", "top_cam",
            "--slp", str(SLP_DIR / "top_cam.slp"),
            "--output", str(TOP_TRACKS),
            "--max-distance", "100",
            "--max-tracks", "3",
            "--max-gap", "500",
        ],
    )
else:
    print("Top tracking skipped. Set RUN_TOP_TRACKS = True")
    print("expected:", TOP_TRACKS)

## 6. 3D reconstruction + bind

**Always pass `--num-candidates 6`.** tqdm must say `3D best-assign`.
If it says `3D candidates`, stop — that is the slow full-permutation path (~9.7 h) and a different algorithm.

Use `--calibration-backend auto` (checked-in refractive model).
Do **not** pass `--max-residual-per-fish 5000` (legacy polynomial gate).

In [ ]:
def recon_args(*, start=None, end=None, output_dir: Path, tag: str) -> list[str]:
    args = [
        EXP,
        "--paths-config", "configs/paths.local.toml",
        "--sleap-output-dir", str(SLP_ROOT),
        "--sleap-subdir", ROUND_TAG,
        "--calibration-backend", "auto",
        "--num-candidates", "6",
        "--use-visible-node-centroid",
        "--xyz-bounds", "-20", "220",
        "--no-shift-to-positive",
        "--max-jump-mm", "8",
        "--max-jump-hard-cap-mm", "80",
        "--max-gap-frames", "100",
        "--num-individuals", "3",
        "--bind-top-id",
        "--top-tracks-csv", str(SLP_DIR / "top_cam.tracks.csv"),
        "--output-dir", str(output_dir),
        "--output-tag", tag,
    ]
    if start is not None:
        args += ["--start-frame", str(start)]
    if end is not None:
        args += ["--end-frame", str(end)]
    return args


if RUN_RECON_SMOKE:
    smoke_dir = OUT_DIR / "smoke_0-4999"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    run_mod(
        "scripts.reconstruction_3d.reconstruct_from_slp",
        recon_args(start=0, end=RECON_SMOKE_END, output_dir=smoke_dir, tag=f"{ROUND_TAG}_smoke"),
    )
elif RUN_RECON_FULL:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    run_mod(
        "scripts.reconstruction_3d.reconstruct_from_slp",
        recon_args(output_dir=OUT_DIR, tag=ROUND_TAG),
    )
else:
    print("Reconstruction skipped. Set RUN_RECON_SMOKE or RUN_RECON_FULL.")
    print("expected CSV:", OUT_DIR / f"{ROUND_TAG}_3d_identified.csv")

## 7. QC + viewer

From `*_best_pre_link.json`: calibration provenance, all-3-fish-valid coverage, residual distribution, `synthetic_views`.

From `*_3d_identified.csv`: speed spikes, `bind_source` (`top_cam_2d` vs `geometry_only`), `bind_score`, stable `final_fish_id`.

Viewer (needs `uv sync --extra rerun`):

```powershell
.venv\Scripts\python.exe -m scripts.visualization.rerun_viewer 20260106_E_2_1_FCMC `
  --paths-config configs/paths.local.toml `
  --sleap-output-dir Z:\medaka\sleap_output `
  --reconstruction-output-dir Z:\medaka\reconstruction_3d `
  --medaka-data-dir Z:\medaka\medaka_202512 `
  --tag round8_260723_strictmouth `
  --slp-dir Z:\medaka\sleap_output\20260106_E_2_1_FCMC\round8_260723_strictmouth `
  --csv Z:\medaka\reconstruction_3d\20260106_E_2_1_FCMC\round8_260723_strictmouth\round8_260723_strictmouth_3d_identified.csv `
  --pre-link-json Z:\medaka\reconstruction_3d\20260106_E_2_1_FCMC\round8_260723_strictmouth\round8_260723_strictmouth_best_pre_link.json `
  --start-frame 0 --end-frame 4999
```

## 8. Optional SAM2 identity re-bind

Use only if `top_cam.tracks.csv` swaps through occlusions. Geometry still comes from Stage 6.

```powershell
.venv\Scripts\python.exe -m sam2_id_experiment.scripts.24_seed_prompts_from_3d `
  --identified-3d-csv Z:\medaka\reconstruction_3d\20260106_E_2_1_FCMC\round8_260723_strictmouth\round8_260723_strictmouth_3d_identified.csv `
  --best-pre-link-json Z:\medaka\reconstruction_3d\20260106_E_2_1_FCMC\round8_260723_strictmouth\round8_260723_strictmouth_best_pre_link.json `
  --paths-config configs/paths.local.toml `
  --output-dir Z:\medaka\runs\seg_id\20260106_E_2_1_FCMC\round8_260723_strictmouth\seed_from_3d
```

Then run `sam2_id_experiment/scripts/01_*` … `23_*`, export `top_cam.seg_identity_tracks.csv`, and re-run Stage 6 with `--top-tracks-csv` pointing at that file.

## Pipeline

```
raw videos (3 cams, 348600 frames)
        │
        ▼
run_round_inference  --max-instances 6 --mode track
        │
        ▼
<view>.slp  (round8_260723_strictmouth)
        │
        ├─► visualize_predictions
        ├─► track_simple_nn top_cam  → top_cam.tracks.csv
        │
        └─► reconstruct_from_slp
              --num-candidates 6          (must see "3D best-assign")
              --use-visible-node-centroid
              --bind-top-id
                │
                ▼
              *_3d.csv
              *_3d_identified.csv
              *_best_pre_link.json
```

## Gotchas

1. `--num-candidates 6` is required for `--max-instances 6` `.slp`.
2. `--use-visible-node-centroid` is the production 3D point.
3. `--max-jump-hard-cap-mm 80` stops a long NaN gap from adopting another fish.
4. Do not carry `--max-residual-per-fish 5000` into a refractive run.
5. Local inference writes merged `.slp`; merge-chunks is only for HPC round7 chunks.